# GTSRB NOTEBOOK - COMPUTER VISION PROJECT
Note: This project ist based on a currently implemented computer vison pipeline Vision-Studio which is currated by myself to speedup the process of define train and ship models.
Repository can be found under: https://github.com/mozi30/CV_VisionStudio

In [ ]:
from typing import Any
import torch
import torch.nn as nn
import torch.nn.functional as f
from torchvision.models import resnet50, swin_t, swin_s, swin_b, swin_v2_t, swin_v2_s, swin_v2_b
from torchvision.models import ResNet50_Weights, Swin_T_Weights, Swin_S_Weights, Swin_B_Weights, Swin_V2_T_Weights, Swin_V2_S_Weights, Swin_V2_B_Weights
from enum import Enum
import cv2
import numpy as np
from typing import Any
from torch.optim import SGD, AdamW
from vision_studio.models import BaseModel, InputSpec, OutputSpec
from vision_studio.types import ClassificationPostprocessOutput, LossOutput
from vision_studio.evaluate import ClassificationEvaluator, print_evaluation_metrics
from vision_studio.dataset import ImageNetClassificationDataset
from vision_studio.data_loader import SimpleDataLoader, BalancedDataLoader
from vision_studio.trainer import WandbTrainer
from vision_studio.augmentation import Resize,Compose,SobelFilter, ContrastNormalization, BrightnessNormalization, HistogramEqualization, Augmentation, EdgeSharpen
from vision_studio.visualise import show_dataset_images, show_dataset_label_distribution

from vision_studio.augmentation.image_augmentations import Brightness, Contrast, GaussianBlur, RandomResizedCrop, Rotate, Saturation

## Models used for image classification

We evaluate three architectures to explore both traditional CNN and modern transformer approaches. **ResNet50** serves as a strong convolutional baseline with pre-trained ImageNet weights. **SwinTransformer** represents state-of-the-art vision transformers with shifted window attention for efficient feature extraction. **CustomModel** is a hybrid design combining local CNN feature extraction with global transformer reasoning, enabling us to test whether architectural innovation offers practical benefits for traffic sign classification. All models are trained under identical conditions to ensure fair comparison.

In [ ]:
class ResNet50(BaseModel):
    def __init__(self, num_classes: int, weights: ResNet50_Weights | None):
        super().__init__()
        self.num_classes = num_classes

        self.model = resnet50(weights=weights)

        model_in_features = self.model.fc.in_features
        self.model.fc = nn.Linear(model_in_features, num_classes)
        self.lossIsWeighted = False

    @property
    def input_spec(self) -> InputSpec:
        """Expected input specification."""
        return InputSpec(dtype=torch.float32)

    @property
    def output_spec(self) -> OutputSpec:
        """Output specification for forward()."""
        return OutputSpec(dtype=torch.float32)

    def forward(self, inputs):
        """Return ONLY logits for speed."""
        logits = self.model(inputs)
        return logits

    def postprocess(self, logits) -> ClassificationPostprocessOutput:
        """Task-specific postprocessing for classification."""
        probs = torch.softmax(logits, dim=1)
        labels = torch.argmax(logits, dim=1)

        return ClassificationPostprocessOutput(
            logits=logits,
            probs=probs,
            labels=labels,
        )

    def set_loss_weights(self, class_weights: torch.Tensor):
        """Set class weights for the loss function."""
        self.lossIsWeighted = True
        self.class_weights = class_weights

    def compute_loss(self, logits, targets) -> LossOutput:
        labels = targets["label"]
        if self.lossIsWeighted:
            loss = f.cross_entropy(logits, labels, weight=self.class_weights.to(logits.device))
        else:
            loss = f.cross_entropy(logits, labels)
        return LossOutput(loss=loss)

In [ ]:


class SwinModelSelect(Enum):
    SWIN_T = 'SWIN_T'
    SWIN_S = 'SWIN_S'
    SWIN_B = 'SWIN_B'
    SWIN_V2_T = 'SWIN_V2_T'
    SWIN_V2_S = 'SWIN_V2_S'
    SWIN_V2_B = 'SWIN_V2_B'

class SwinTransformer(BaseModel):
    def __init__(self, model_select: SwinModelSelect, num_classes: int, weights: Any):
        super().__init__()
        self.num_classes = num_classes

        if model_select == SwinModelSelect.SWIN_T:
            if weights is not None and not isinstance(weights, Swin_T_Weights):
                raise ValueError(f"Expected weights to be of type Swin_T_Weights, got {type(weights)}")
            self.model = swin_t(weights=weights)
        elif model_select == SwinModelSelect.SWIN_S:
            if weights is not None and not isinstance(weights, Swin_S_Weights):
                raise ValueError(f"Expected weights to be of type Swin_S_Weights, got {type(weights)}")
            self.model = swin_s(weights=weights)
        elif model_select == SwinModelSelect.SWIN_B:
            if weights is not None and not isinstance(weights, Swin_B_Weights):
                raise ValueError(f"Expected weights to be of type Swin_B_Weights, got {type(weights)}")
            self.model = swin_b(weights=weights)
        elif model_select == SwinModelSelect.SWIN_V2_T:
            if weights is not None and not isinstance(weights, Swin_V2_T_Weights):
                raise ValueError(f"Expected weights to be of type Swin_V2_T_Weights, got {type(weights)}")
            self.model = swin_v2_t(weights=weights)
        elif model_select == SwinModelSelect.SWIN_V2_S:
            if weights is not None and not isinstance(weights, Swin_V2_S_Weights):
                raise ValueError(f"Expected weights to be of type Swin_V2_S_Weights, got {type(weights)}")
            self.model = swin_v2_s(weights=weights)
        elif model_select == SwinModelSelect.SWIN_V2_B:
            if weights is not None and not isinstance(weights, Swin_V2_B_Weights):
                raise ValueError(f"Expected weights to be of type Swin_V2_B_Weights, got {type(weights)}")
            self.model = swin_v2_b(weights=weights)

        incoming_signal_size = self.model.head.in_features
        self.model.head = nn.Linear(incoming_signal_size, num_classes)

    @property
    def input_spec(self) -> InputSpec:
        """Expected input specification."""
        return InputSpec(dtype=torch.float32)

    @property
    def output_spec(self) -> OutputSpec:
        """Output specification for forward()."""
        return OutputSpec(dtype=torch.float32)

    def forward(self, inputs):
        """Return ONLY logits for speed."""
        _, _, h, w = inputs.shape

        if h % 4 != 0 or w % 4 != 0:
            raise ValueError(f"Input size must be divisible by 4, got {h}x{w}")
        logits = self.model(inputs)
        return logits

    def postprocess(self, logits) -> ClassificationPostprocessOutput:
        """Task-specific postprocessing for classification."""
        probs = torch.softmax(logits, dim=1)
        labels = torch.argmax(logits, dim=1)

        return ClassificationPostprocessOutput(
            logits=logits,
            probs=probs,
            labels=labels,
        )

    def compute_loss(self, logits, targets) -> LossOutput:
        """Compute loss from raw logits."""
        labels = targets["label"]
        loss = f.cross_entropy(logits, labels)
        return LossOutput(loss=loss)

In [ ]:
class GlobalTransformerBlock(nn.Module):
    def __init__(self, channels: int, heads: int = 4, layers: int = 1):
        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=channels,
            nhead=heads,
            dim_feedforward=channels * 2,
            dropout=0.1,
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=layers,
        )

    def forward(self, x):
        # x: (B, C, H, W)
        B, C, H, W = x.shape

        tokens = x.flatten(2).transpose(1, 2)   # (B, H*W, C)
        tokens = self.transformer(tokens)
        x = tokens.transpose(1, 2).view(B, C, H, W)

        return x


class CustomModel(BaseModel):
    def __init__(self, input_shape: tuple, num_classes: int):
        super().__init__()

        self.num_classes = num_classes

        self.features = nn.Sequential(
            # 224 -> 112
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # 112 -> 56
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # 56 -> 28
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # 28 -> 14
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        self.global_attn = GlobalTransformerBlock(
            channels=128,
            heads=4,
            layers=1,
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )
        self.lossIsWeighted = False

    @property
    def input_spec(self) -> InputSpec:
        return InputSpec(dtype=torch.float32)

    @property
    def output_spec(self) -> OutputSpec:
        return OutputSpec(dtype=torch.float32)

    def forward(self, inputs):
        x = self.features(inputs)      # (B, 128, 14, 14) for 224x224
        x = self.global_attn(x)        # global attention over 196 tokens
        logits = self.classifier(x)
        return logits

    def postprocess(self, logits) -> ClassificationPostprocessOutput:
        probs = torch.softmax(logits, dim=1)
        labels = torch.argmax(logits, dim=1)

        return ClassificationPostprocessOutput(
            logits=logits,
            probs=probs,
            labels=labels,
        )
    
    def set_loss_weights(self, class_weights: torch.Tensor):
        """Set class weights for the loss function."""
        self.lossIsWeighted = True
        self.class_weights = class_weights

    def compute_loss(self, logits, targets) -> LossOutput:
        labels = targets["label"]
        if self.lossIsWeighted:
            loss = f.cross_entropy(logits, labels, weight=self.class_weights.to(logits.device))
        else:
            loss = f.cross_entropy(logits, labels)
        return LossOutput(loss=loss)

## Define augmentations and image preprocessing

Custom augmentations and preprocessing pipelines are designed to enhance the visual distinctiveness of traffic signs and improve robustness to real-world variations. A custom `BoostTrafficSignColors` augmentation selectively amplifies red, blue, and yellow hues—the dominant colors in German traffic signs—by boosting saturation and brightness in HSV space. Beyond color boosting, we test additional preprocessing strategies including histogram equalization for improved contrast, contrast and brightness normalization for lighting invariance, Sobel edge detection for boundary emphasis, and edge sharpening. Each augmentation is composed with resizing and evaluated independently to measure its impact on model generalization and class-specific performance across the GTSRB dataset.

In [ ]:
class BoostTrafficSignColors(Augmentation):
    def __init__(
        self,
        saturation_gain: float = 1.4,
        value_gain: float = 1.05,
        p: float = 1.0,
    ):
        super().__init__(p=p)
        self.saturation_gain = saturation_gain
        self.value_gain = value_gain

    def __call__(
        self,
        image: np.ndarray,
        target: dict[str, Any],
    ) -> tuple[np.ndarray, dict[str, Any]]:

        img = image.copy()
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)

        h, s, v = cv2.split(hsv)

        # --- Define color masks (German traffic sign colors) ---

        # Red (two ranges)
        lower_red1 = np.array([0, 100, 100])
        upper_red1 = np.array([10, 255, 255])
        lower_red2 = np.array([160, 100, 100])
        upper_red2 = np.array([180, 255, 255])

        mask_red = cv2.inRange(hsv, lower_red1, upper_red1) | \
                   cv2.inRange(hsv, lower_red2, upper_red2)

        # Blue
        lower_blue = np.array([90, 100, 100])
        upper_blue = np.array([130, 255, 255])
        mask_blue = cv2.inRange(hsv, lower_blue, upper_blue)

        # Yellow
        lower_yellow = np.array([15, 100, 100])
        upper_yellow = np.array([35, 255, 255])
        mask_yellow = cv2.inRange(hsv, lower_yellow, upper_yellow)

        # Combine masks
        mask = (mask_red | mask_blue | mask_yellow) > 0

        # --- Apply boost ONLY where mask is true ---
        s = s.astype(np.float32)
        v = v.astype(np.float32)

        s[mask] *= self.saturation_gain
        v[mask] *= self.value_gain

        s = np.clip(s, 0, 255).astype(np.uint8)
        v = np.clip(v, 0, 255).astype(np.uint8)

        hsv_boosted = cv2.merge([h, s, v])
        boosted = cv2.cvtColor(hsv_boosted, cv2.COLOR_HSV2RGB)

        return boosted.astype(image.dtype), target
        

In [ ]:
resize = Resize(224, 224)

hist_eqal = Compose([
    Resize(224, 224),
    HistogramEqualization(),
])

normalisation = Compose([
    Resize(224, 224),
    ContrastNormalization(),
    BrightnessNormalization(),
])

sobel = Compose([
    Resize(224, 224),
    SobelFilter(),
])

traffic_sign_boost = Compose([
    Resize(224, 224),
    EdgeSharpen(strength=2, sigma=14.0),
    BrightnessNormalization(),
    ContrastNormalization(),
    BoostTrafficSignColors(saturation_gain=1.2, value_gain=1.02)
])

strengthen_edges = Compose([
    Resize(224, 224),
    EdgeSharpen(strength=2, sigma=7.0)
])

train_augmentations = Compose([
    Resize(224, 224), 
    RandomResizedCrop(224,224, scale=(0.9, 1.0)), 
    Brightness(0.75, 1.25, p=0.4),
    Contrast(0.85, 1.25, p=0.4),
    Saturation(0.9, 1.3, p=0.5),
    GaussianBlur(0.1, 0.8, p=0.2),
    Rotate(5, p=0.3),
    Rotate(-5, p=0.3),
])

In [ ]:
dataset_path = "/home/mozi30/data/datasets/gtsrb_out"
dataset_train = ImageNetClassificationDataset(root_dir=dataset_path, transform=resize)
dataset_val = ImageNetClassificationDataset(root_dir=dataset_path, split="val", transform=resize)
dataset_train_sobel = ImageNetClassificationDataset(root_dir=dataset_path, transform=sobel)
dataset_val_sobel = ImageNetClassificationDataset(root_dir=dataset_path, split="val", transform=sobel)
dataset_train_hist_eqal = ImageNetClassificationDataset(root_dir=dataset_path, transform=hist_eqal)
dataset_val_hist_eqal = ImageNetClassificationDataset(root_dir=dataset_path, split="val", transform=hist_eqal)
dataset_train_normalisation = ImageNetClassificationDataset(root_dir=dataset_path, transform=normalisation)
dataset_val_normalisation = ImageNetClassificationDataset(root_dir=dataset_path, split="val", transform=normalisation)
dataset_train_traffic_boost = ImageNetClassificationDataset(root_dir=dataset_path, transform=traffic_sign_boost)
dataset_val_traffic_boost = ImageNetClassificationDataset(root_dir=dataset_path, split="val", transform=traffic_sign_boost)
dataset_train_edge_boost = ImageNetClassificationDataset(root_dir=dataset_path, transform=strengthen_edges)
dataset_val_edge_boost = ImageNetClassificationDataset(root_dir=dataset_path, split="val", transform=strengthen_edges)
dataset_train_augmented = ImageNetClassificationDataset(root_dir=dataset_path, transform=train_augmentations)
dataset_val_augmented = ImageNetClassificationDataset(root_dir=dataset_path, split="val", transform=resize)

dataset_dict = {
    #"base": (dataset_train, dataset_val),
    "sobel": (dataset_train_sobel, dataset_val_sobel),
    "hist_eqal": (dataset_train_hist_eqal, dataset_val_hist_eqal),
    "normalisation": (dataset_train_normalisation, dataset_val_normalisation),
    "traffic_boost": (dataset_train_traffic_boost, dataset_val_traffic_boost),
    "edge_boost_str_2_sig_7_0": (dataset_train_edge_boost, dataset_val_edge_boost),
}
myModel_dataset_dict = {
    "base": (dataset_train, dataset_val),
    "sobel": (dataset_train_sobel, dataset_val_sobel),
    "hist_eqal": (dataset_train_hist_eqal, dataset_val_hist_eqal),
    "normalisation": (dataset_train_normalisation, dataset_val_normalisation),
    "traffic_boost": (dataset_train_traffic_boost, dataset_val_traffic_boost),
    "edge_boost_str_2_sig_7_0": (dataset_train_edge_boost, dataset_val_edge_boost),
    "augmented": (dataset_train_augmented, dataset_val_augmented)
}


In [ ]:
print("Base visualisation")
show_dataset_images(dataset_train, )
show_dataset_images(dataset_val, shuffle=True)
print("Sobel visualisation")
show_dataset_images(dataset_train_sobel, shuffle=True)
show_dataset_images(dataset_val_sobel, shuffle=True)
print("Histogram Equalisation visualisation")
show_dataset_images(dataset_train_hist_eqal, shuffle=True)
show_dataset_images(dataset_val_hist_eqal, shuffle=True)
print("Contrast and Brightness Normalisation visualisation")
show_dataset_images(dataset_train_normalisation, shuffle=True)
show_dataset_images(dataset_val_normalisation, shuffle=True)
print("Traffic Sign Boost visualisation")
show_dataset_images(dataset_train_traffic_boost, shuffle=True)
show_dataset_images(dataset_val_traffic_boost, shuffle=True)
print("Edge Boost visualisation")
show_dataset_images(dataset_train_edge_boost, )
show_dataset_images(dataset_val_edge_boost, shuffle=True)
print("Augmented visualisation")
show_dataset_images(dataset_train_augmented, shuffle=True)
show_dataset_images(dataset_val_augmented, shuffle=True)

## Train Models

Base training creates a systematic grid of model runs across all augmentation variants and optimizer choices. Each model (ResNet50 and CustomModel) is trained on every augmentation strategy (base, Sobel, histogram equalization, normalization, traffic sign boost, edge sharpening) using both SGD and AdamW optimizers for 100 epochs with validation tracking via Weights & Biases. To address class imbalance detected in the GTSRB dataset, we incorporate weighted loss and balanced sampling strategies into selected runs. Checkpoints are saved after each run to enable later fine-tuning and facilitate comparative analysis across the full experimental grid. This systematic approach ensures reproducibility and provides a foundation for identifying the most promising model and augmentation combinations.

In [ ]:
for name, (train_dataset, val_dataset) in dataset_dict.items():
    data_loader_train = SimpleDataLoader(dataset=train_dataset, dataset_percentage_per_epoch=10, batch_size=96, shuffle=True)
    data_loader_val = SimpleDataLoader(dataset=val_dataset, batch_size=96, shuffle=False)
    print(f"DataLoader for {name} created with {len(data_loader_train)} training batches and {len(data_loader_val)} validation batches.")
    resnet_50 = ResNet50(num_classes=dataset_train.get_num_classes(), weights=ResNet50_Weights.IMAGENET1K_V2)
    sgd_optimizer = AdamW(resnet_50.parameters(), lr=0.001)
    sgd_optimizer = SGD(resnet_50.parameters(), lr=0.01, momentum=0.9)

    # Create evaluator with type-safe metrics
    evaluator = ClassificationEvaluator(
        num_classes=dataset_train.get_num_classes(),
        topk=(1, 5)
    )
    # Create trainer with evaluator
    run_name = f"resnet50-classification-{name}"
    trainer = WandbTrainer(
        optimizer=sgd_optimizer,
        evaluator=evaluator,
        epochs=20,
        project="cv-course",
        run_name=run_name,
        device="cuda"
    )
    trainer.fit(model=resnet_50, train_loader=data_loader_train, val_loader=data_loader_val)
    trainer.save_checkpoint(f"/home/mozi30/data/models/resnet50-{name}", model = resnet_50)

In [ ]:
for name, (train_dataset, val_dataset) in dataset_dict.items():
    data_loader_train = SimpleDataLoader(dataset=train_dataset, dataset_percentage_per_epoch=10, batch_size=32, shuffle=True)
    data_loader_val = SimpleDataLoader(dataset=val_dataset, batch_size=32, shuffle=False)
    print(f"DataLoader for {name} created with {len(data_loader_train)} training batches and {len(data_loader_val)} validation batches.")
    swinTransformer = SwinTransformer(model_select=SwinModelSelect.SWIN_V2_T, num_classes=dataset_train.get_num_classes(), weights=Swin_V2_T_Weights.IMAGENET1K_V1)
    adam_optimizer = AdamW(swinTransformer.parameters(), lr=5e-5, weight_decay=0.05)
    sgd_optimizer = SGD(swinTransformer.parameters(), lr=0.01, momentum=0.9)

    # Create evaluator with type-safe metrics
    evaluator = ClassificationEvaluator(
        num_classes=dataset_train.get_num_classes(),
        topk=(1, 5)
    )
    # Create trainer with evaluator
    run_name = f"swintransformer-classification-{name}"
    trainer = WandbTrainer(
        optimizer=adam_optimizer,
        evaluator=evaluator,
        epochs=20,
        project="cv-course",
        run_name=run_name,
        device="cuda"
    )
    trainer.fit(model=swinTransformer, train_loader=data_loader_train, val_loader=data_loader_val)
    trainer.save_checkpoint(f"/home/mozi30/data/models/swintransformer-{name}.pth", model = swinTransformer)

In [ ]:
for name, (train_dataset, val_dataset) in myModel_dataset_dict.items():
    data_loader_train = SimpleDataLoader(dataset=train_dataset, dataset_percentage_per_epoch=10, batch_size=64, shuffle=True)
    data_loader_val = SimpleDataLoader(dataset=val_dataset,dataset_percentage_per_epoch=10, batch_size=64, shuffle=False)
    print(f"DataLoader for {name} created with {len(data_loader_train)} training batches and {len(data_loader_val)} validation batches.")
    myModel = CustomModel(input_shape=(3, 224, 224), num_classes=dataset_train.get_num_classes())
    adam_optimizer = AdamW(myModel.parameters(), lr=0.001)
    sgd_optimizer = SGD(myModel.parameters(), lr=0.01, momentum=0.9)

    # Create evaluator with type-safe metrics
    evaluator = ClassificationEvaluator(
        num_classes=dataset_train.get_num_classes(),
        topk=(1, 5)
    )
    # Create trainer with evaluator
    run_name = f"myModel-classification-{name}"
    trainer = WandbTrainer(
        optimizer=adam_optimizer,
        evaluator=evaluator,
        epochs=100,
        project="cv-course",
        run_name=run_name,
        device="cuda"
    )
    trainer.fit(model=myModel, train_loader=data_loader_train, val_loader=data_loader_val)
    trainer.save_checkpoint(f"/home/mozi30/data/models/myModel-{name}.pth", model = myModel)

## Fine tune models

Fine-tuning refines previously trained models using reduced learning rates over shorter schedules (5–20 epochs depending on the model). We load saved checkpoints from initial training and continue optimization with conservative learning rates (0.0001 for AdamW, 0.001 for SGD) to adapt pre-trained features without catastrophic forgetting. Both ResNet50 and CustomModel checkpoints are fine-tuned on all augmentation variants, with separate trials for each optimizer type. Results from fine-tuning are compared against base training outcomes to quantify genuine improvements and identify which configurations benefit most from extended optimization. Checkpoints are saved throughout to preserve gains and enable model selection based on validation performance.

In [ ]:
for name, (train_dataset, val_dataset) in dataset_dict.items():
    finetune_loader_train = SimpleDataLoader(dataset=train_dataset, batch_size=96, shuffle=True)
    finetune_loader_val = SimpleDataLoader(dataset=val_dataset, batch_size=96, shuffle=False)
    print(f"DataLoader for {name} created with {len(finetune_loader_train)} training batches and {len(finetune_loader_val)} validation batches.")
    resnet_50 = ResNet50(num_classes=dataset_train.get_num_classes(), weights=ResNet50_Weights.IMAGENET1K_V2)
    resnet_50.load_checkpoint(f"/home/mozi30/data/models/resnet50-{name}")
    adamW_optimizer = AdamW(resnet_50.parameters(), lr=0.0001)
    evaluator = ClassificationEvaluator(
        num_classes=dataset_train.get_num_classes(),
        topk=(1, 5)
    )
    trainer = WandbTrainer(
        optimizer=adamW_optimizer,
        evaluator=evaluator,
        epochs=5,
        project="cv-course",
        run_name=f"resnet50-finetune-{name}",
        device="cuda"
    )
    trainer.fit(model=resnet_50, train_loader=finetune_loader_train, val_loader=finetune_loader_val)
    trainer.save_checkpoint(f"/home/mozi30/data/models/resnet50-finetune-{name}", model = resnet_50)

    resnet_50 = ResNet50(num_classes=dataset_train.get_num_classes(), weights=ResNet50_Weights.IMAGENET1K_V2)
    resnet_50.load_checkpoint(f"/home/mozi30/data/models/resnet50-finetune-{name}")
    sgd_optimizer = SGD(resnet_50.parameters(), lr=0.001, momentum=0.9)
    trainer = WandbTrainer(
        optimizer=sgd_optimizer,
        evaluator=evaluator,
        epochs=5,
        project="cv-course",
        run_name=f"resnet50-sgd-finetune-{name}",
        device="cuda"
    )
    trainer.fit(model=resnet_50, train_loader=finetune_loader_train, val_loader=finetune_loader_val)
    trainer.save_checkpoint(f"/home/mozi30/data/models/resnet50-sgd-finetune-{name}.pth", model = resnet_50)

In [ ]:
for name, (train_dataset, val_dataset) in myModel_dataset_dict.items():
    finetune_loader_train = SimpleDataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
    finetune_loader_val = SimpleDataLoader(dataset=val_dataset, batch_size=64, shuffle=False)
    print(f"DataLoader for {name} created with {len(finetune_loader_train)} training batches and {len(finetune_loader_val)} validation batches.")

    myModel = CustomModel(input_shape=(3, 224, 224), num_classes=dataset_train.get_num_classes())
    myModel.load_checkpoint(f"/home/mozi30/data/models/myModel-{name}.pth")

    adamW_optimizer = AdamW(myModel.parameters(), lr=0.0001)
    trainer = WandbTrainer(
        optimizer=adamW_optimizer,
        evaluator=evaluator,
        epochs=20,
        project="cv-course",
        run_name=f"myModel-finetune-{name}",
        device="cuda"
    )
    trainer.fit(model=myModel, train_loader=finetune_loader_train, val_loader=finetune_loader_val)

    trainer.save_checkpoint(f"/home/mozi30/data/models/myModel-finetune-{name}.pth", model=myModel)


    myModel = CustomModel(input_shape=(3, 224, 224), num_classes=dataset_train.get_num_classes())
    myModel.load_checkpoint(f"/home/mozi30/data/models/myModel-finetune-{name}.pth")

    sgd_optimizer = SGD(myModel.parameters(), lr=0.001, momentum=0.9)
    trainer = WandbTrainer(
        optimizer=sgd_optimizer,
        evaluator=evaluator,
        epochs=20,
        project="cv-course",
        run_name=f"myModel-sgd-finetune-{name}",
        device="cuda"
    )
    trainer.fit(model=myModel, train_loader=finetune_loader_train, val_loader=finetune_loader_val)
    trainer.save_checkpoint(f"/home/mozi30/data/models/myModel-sgd-finetune-{name}.pth", model=myModel)


## Class Distribution Deepdive

This section trains specialized models using insights from class distribution analysis, focusing on balanced sampling and loss weighting to mitigate class imbalance. A CustomModel is trained with augmentation and balanced data loading, then fine-tuned with both AdamW and SGD to optimize convergence. Class sample counts are computed and used to derive per-class loss weights (inverse frequency), which are then applied during training to penalize misclassification of under-represented sign types more heavily. Multiple training runs—with and without weighted loss, with and without balanced sampling—enable isolation of each technique's contribution to per-class robustness and overall accuracy. Results inform the selection of the best combination of balancing strategies for final model training.

In [ ]:
train_augmentations = Compose([
    Resize(224, 224), 
    RandomResizedCrop(224,224, scale=(0.9, 1.0)), 
    Brightness(0.75, 1.25, p=0.4),
    Contrast(0.85, 1.25, p=0.4),
    Saturation(0.9, 1.3, p=0.5),
    GaussianBlur(0.1, 0.8, p=0.2),
    Rotate(5, p=0.2),
    Rotate(-5, p=0.2),
    Rotate(5, p=0.2),
    Rotate(-5, p=0.2),
    Rotate(5, p=0.2),
    Rotate(-5, p=0.2),
    Rotate(5, p=0.2),
    Rotate(-5, p=0.2),
])
dataset_train = ImageNetClassificationDataset(root_dir=dataset_path, transform=train_augmentations)
dataset_val = ImageNetClassificationDataset(root_dir=dataset_path, split="val", transform=resize)

balanced_data_loader_train = BalancedDataLoader(dataset=dataset_train, batch_size=64, shuffle=True)
data_loader_val = SimpleDataLoader(dataset=dataset_val, batch_size=64, shuffle=False)
print(f"DataLoader for augmented dataset created with {len(balanced_data_loader_train)} training batches and {len(data_loader_val)} validation batches.")

In [ ]:
myModel = CustomModel(input_shape=(3, 224, 224), num_classes=dataset_train.get_num_classes())
optimizer = AdamW(myModel.parameters(), lr=0.001)
finetine_optimizer = AdamW(myModel.parameters(), lr=0.0001)
evaluator = ClassificationEvaluator(
    num_classes=dataset_train.get_num_classes(),
    topk=(1, 5)
)
trainer = WandbTrainer(
    optimizer=optimizer,
    evaluator=evaluator,
    epochs=50,
    project="cv-course",
    run_name=f"myModel-augmented-balanced",
    device="cuda"
)
trainer.fit(model=myModel, train_loader=data_loader_train, val_loader=data_loader_val)
trainer.save_checkpoint(f"/home/mozi30/data/models/myModel-augmented-balanced.pth", model=myModel)
trainer = WandbTrainer(
    optimizer=finetine_optimizer,
    evaluator=evaluator,
    epochs=60,
    project="cv-course",
    run_name=f"myModel-augmented-balanced-finetune",
    device="cuda"
)
trainer.load_checkpoint(f"/home/mozi30/data/models/myModel-augmented-balanced.pth", model=myModel)
trainer.fit(model=myModel, train_loader=balanced_data_loader_train, val_loader=data_loader_val)
trainer.save_checkpoint(f"/home/mozi30/data/models/myModel-augmented-balanced-finetune.pth", model=myModel)



class_sample_counts = dataset_train.get_class_sample_counts()
class_weights = 1. / torch.tensor([class_sample_counts[i] for i in sorted(class_sample_counts.keys())], dtype=torch.float)
myModel.set_loss_weights(class_weights=class_weights.cuda())
trainer = WandbTrainer(
    optimizer=optimizer,
    evaluator=evaluator,
    epochs=50,
    project="cv-course",
    run_name=f"myModel-weighted-loss-augmented-balanced",
    device="cuda"
)
trainer.fit(model=myModel, train_loader=data_loader_train, val_loader=data_loader_val)
trainer.save_checkpoint(f"/home/mozi30/data/models/myModel-weighted-loss-augmented-balanced.pth", model=myModel)
trainer = WandbTrainer(
    optimizer=finetine_optimizer,
    evaluator=evaluator,
    epochs=60,
    project="cv-course",
    run_name=f"myModel-weighted-loss-augmented-balanced-finetune",
    device="cuda"
)
trainer.load_checkpoint(f"/home/mozi30/data/models/myModel-weighted-loss-augmented-balanced.pth", model=myModel)
trainer.fit(model=myModel, train_loader=data_loader_train, val_loader=data_loader_val)
trainer.save_checkpoint(f"/home/mozi30/data/models/myModel-weighted-loss-augmented-balanced-finetune.pth", model=myModel)



## Train the best model possible

Based on findings from initial training, class distribution analysis, and fine-tuning experiments, this section trains a final shortlist of candidate models using the most effective augmentations and tuned hyperparameters. The primary candidate is ResNet50 with histogram equalization combined with dynamic augmentation, weighted loss weighting, and balanced sampling. Training emphasizes per-class robustness and consistent validation gains over single-run peak accuracy, using conservative regularization and extended schedules (10–50 epochs depending on the variant). Each candidate undergoes initial training, fine-tuning with reduced learning rates, and comprehensive logging to Weights & Biases for traceability and reproducibility. Final checkpoints are saved and reserved for extended validation and potential deployment testing.

In [ ]:
from pathlib import Path


def load_state_dict(
    path: str | Path,
    model,
) -> dict[str, Any]:
    checkpoint = torch.load(str(path), map_location="cuda" if torch.cuda.is_available() else "cpu")
    model.load_state_dict(checkpoint["model_state_dict"])
    return checkpoint

In [ ]:
train_augmentation_hist_equal = Compose([
    Resize(224, 224), 
    HistogramEqualization(),
    RandomResizedCrop(224,224, scale=(0.9, 1.0)), 
    Brightness(0.75, 1.25, p=0.4),
    Contrast(0.85, 1.25, p=0.4),
    Saturation(0.9, 1.3, p=0.5),
    GaussianBlur(0.1, 0.8, p=0.2),
    Rotate(5, p=0.3),
    Rotate(-5, p=0.3),
])
dataset_train_aug_hist_equal = ImageNetClassificationDataset(root_dir=dataset_path, transform=train_augmentation_hist_equal)
#balanced_loader = BalancedDataLoader(dataset=dataset_train_aug_hist_equal, batch_size=64, shuffle=True)
val_loader = SimpleDataLoader(dataset=dataset_val_hist_eqal, batch_size=64, shuffle=False)
myModel = ResNet50(num_classes=dataset_train_aug_hist_equal.get_num_classes(), weights=ResNet50_Weights.IMAGENET1K_V2)
finetune_optimizer = SGD(myModel.parameters(), lr=0.001, momentum=0.9)
evaluator = ClassificationEvaluator(
    num_classes=dataset_train_aug_hist_equal.get_num_classes(),
    topk=(1, 5)
)

In [ ]:
trainer = WandbTrainer(
    optimizer=finetine_optimizer,
    evaluator=evaluator,
    epochs=10,
    project="cv-course",
    run_name=f"resnet50-finetune-hist_equal-augmented-balanced",
    device="cuda"
)
load_state_dict(f"/home/mozi30/data/models/resnet50-sgd-finetune-hist_eqal.pth", model=myModel)
trainer.fit(model=myModel, train_loader=balanced_loader, val_loader=val_loader)
trainer.save_checkpoint(f"/home/mozi30/data/models/resnet50-finetune-hist_equal-augmented-balanced-finetune.pth", model=myModel)

train_loader = SimpleDataLoader(dataset=dataset_train_aug_hist_equal, batch_size=64, shuffle=True)
class_sample_counts = dataset_train_aug_hist_equal.get_class_sample_counts()
class_weights = 1. / torch.tensor([class_sample_counts[i] for i in sorted(class_sample_counts.keys())], dtype=torch.float)
myModel.set_loss_weights(class_weights=class_weights.cuda())
trainer = WandbTrainer(
    optimizer=finetine_optimizer,
    evaluator=evaluator,
    epochs=10,
    project="cv-course",
    run_name=f"resnet50-finetune-hist_equal-augmented-weighted-loss",
    device="cuda"
)
load_state_dict(f"/home/mozi30/data/models/resnet50-sgd-finetune-hist_eqal.pth", model=myModel)
trainer.fit(model=myModel, train_loader=train_loader, val_loader=val_loader)
trainer.save_checkpoint(f"/home/mozi30/data/models/resnet50-finetune-hist_equal-augmented-weighted-loss.pth", model=myModel)



## Validate models

Comprehensive validation evaluates all trained models from the experimental grid on held-out test data, reporting both aggregate metrics (overall accuracy, macro and micro F1) and per-class measures (precision, recall, F1). Models are loaded from saved checkpoints and run in evaluation mode on the full validation set without augmentation to measure generalization. A `model_lookup` dictionary organizes all model paths, augmentation types, and weight configurations, enabling systematic evaluation of hundreds of model variants in a single pass. Results are stored in a nested dictionary keyed by (model_name, augmentation, weight_type) for later aggregation and analysis. This systematic approach surfaces class-specific weaknesses, detects overfitting, and supports objective model selection based on reliable metrics across the full experimental matrix.

In [ ]:
model_lookup = {
    "resnet": {
        "base": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/resnet50-base.pth",
                "finetune": "/home/mozi30/data/models/resnet50-finetune-base.pth",
                "sgd_finetune": "/home/mozi30/data/models/resnet50-sgd-finetune-base.pth",
            },
            "dataset": (dataset_val),
        },
        "sobel": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/resnet50-sobel.pth",
                "finetune": "/home/mozi30/data/models/resnet50-finetune-sobel.pth",
                "sgd_finetune": "/home/mozi30/data/models/resnet50-sgd-finetune-sobel.pth",
            },
            "dataset": (dataset_val_sobel),
        },
        "hist_eqal": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/resnet50-hist_eqal.pth",
                "finetune": "/home/mozi30/data/models/resnet50-finetune-hist_eqal.pth",
                "sgd_finetune": "/home/mozi30/data/models/resnet50-sgd-finetune-hist_eqal.pth",
            },
            "dataset": (dataset_val_hist_eqal),
        },
        "normalisation": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/resnet50-normalisation.pth",
                "finetune": "/home/mozi30/data/models/resnet50-finetune-normalisation.pth",
                "sgd_finetune": "/home/mozi30/data/models/resnet50-sgd-finetune-normalisation.pth",
            },
            "dataset": (dataset_val_normalisation),
        },
        "traffic_boost": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/resnet50-traffic_boost.pth",
                "finetune": "/home/mozi30/data/models/resnet50-finetune-traffic_boost.pth",
                "sgd_finetune": "/home/mozi30/data/models/resnet50-sgd-finetune-traffic_boost.pth",
            },
            "dataset": (dataset_val_traffic_boost),
        },
        "edge_boost_str_2_sig_7_0": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/resnet50-edge_boost_str_2_sig_7_0.pth",
                "finetune": "/home/mozi30/data/models/resnet50-finetune-edge_boost_str_2_sig_7_0.pth",
                "sgd_finetune": "/home/mozi30/data/models/resnet50-sgd-finetune-edge_boost_str_2_sig_7_0.pth",
            },
            "dataset": (dataset_val_edge_boost),
        },
    },
    "swintransformer": {
        "base": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/swintransformer-base.pth",
            },
            "dataset": (dataset_val),
        },
        "sobel": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/swintransformer-sobel.pth",
            },
            "dataset": (dataset_val_sobel),
        },
        "hist_eqal": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/swintransformer-hist_eqal.pth",
            },
            "dataset": (dataset_val_hist_eqal),
        },
        "normalisation": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/swintransformer-normalisation.pth",
            },
            "dataset": (dataset_val_normalisation),
        },
        "traffic_boost": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/swintransformer-traffic_boost.pth",
            },
            "dataset": (dataset_val_traffic_boost),
        },
        "edge_boost_str_2_sig_7_0": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/swintransformer-edge_boost_str_2_sig_7_0.pth",
            },
            "dataset": (dataset_val_edge_boost),
        },
    },
    "myModel": {
        "base": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/myModel-base.pth",
                "finetune": "/home/mozi30/data/models/myModel-finetune-base.pth",
                "sgd_finetune": "/home/mozi30/data/models/myModel-sgd-finetune-base.pth",
            },
            "dataset": (dataset_val),
        },
        "sobel": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/myModel-sobel.pth",
                "finetune": "/home/mozi30/data/models/myModel-finetune-sobel.pth",
                "sgd_finetune": "/home/mozi30/data/models/myModel-sgd-finetune-sobel.pth",
            },
            "dataset": (dataset_val_sobel),
        },
        "hist_eqal": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/myModel-hist_eqal.pth",
                "finetune": "/home/mozi30/data/models/myModel-finetune-hist_eqal.pth",
                "sgd_finetune": "/home/mozi30/data/models/myModel-sgd-finetune-hist_eqal.pth",
            },
            "dataset": (dataset_val_hist_eqal),
        },
        "normalisation": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/myModel-normalisation.pth",
                "finetune": "/home/mozi30/data/models/myModel-finetune-normalisation.pth",
                "sgd_finetune": "/home/mozi30/data/models/myModel-sgd-finetune-normalisation.pth",
            },
            "dataset": (dataset_val_normalisation),
        },
        "traffic_boost": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/myModel-traffic_boost.pth",
                "finetune": "/home/mozi30/data/models/myModel-finetune-traffic_boost.pth",
                "sgd_finetune": "/home/mozi30/data/models/myModel-sgd-finetune-traffic_boost.pth",
            },
            "dataset": (dataset_val_traffic_boost),
        },
        "edge_boost_str_2_sig_7_0": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/myModel-edge_boost_str_2_sig_7_0.pth",
                "finetune": "/home/mozi30/data/models/myModel-finetune-edge_boost_str_2_sig_7_0.pth",
                "sgd_finetune": "/home/mozi30/data/models/myModel-sgd-finetune-edge_boost_str_2_sig_7_0.pth",
            },
            "dataset": (dataset_val_edge_boost),
        },
        "augmented": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/myModel-augmented.pth",
                "finetune": "/home/mozi30/data/models/myModel-finetune-augmented.pth",
                "sgd_finetune": "/home/mozi30/data/models/myModel-sgd-finetune-augmented.pth",
            },
            "dataset": (dataset_val),
        },
        "augmented-balanced": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/myModel-augmented-balanced.pth",
                "finetune": "/home/mozi30/data/models/myModel-augmented-balanced-finetune.pth",
            },
            "dataset": (dataset_val),
        },
        "weighted-loss-augmented-balanced": {
            "weights": {
                "pre-training": "/home/mozi30/data/models/myModel-weighted-loss-augmented-balanced.pth",
                "finetune": "/home/mozi30/data/models/myModel-weighted-loss-augmented-balanced-finetune.pth",
            },
            "dataset": (dataset_val),
        },
    },
}

model_lookup_best_model = {
    "resnet": {
        "base": {
                "weights": {
                    "sgd_finetune": "/home/mozi30/data/models/resnet50-sgd-finetune-base.pth",
                },
                "dataset": (dataset_val),
        },
        "hist_eqal": {
            "weights": {
                "sgd_finetune": "/home/mozi30/data/models/resnet50-sgd-finetune-hist_eqal.pth",
            },
            "dataset": (dataset_val_hist_eqal),
        },
        "hist_equal_augmented_weighted_loss": {
            "weights": {
                "sgd_finetune": "/home/mozi30/data/models/resnet50-finetune-hist_equal-augmented-weighted-loss.pth",
            },
            "dataset": (dataset_val_hist_eqal),
        },
        "hist_equal_augmented_balanced": {
            "weights": {
                "sgd_finetune": "/home/mozi30/data/models/resnet50-finetune-hist_equal-augmented-balanced-finetune.pth",
            },
            "dataset": (dataset_val_hist_eqal),
        },
    }
}

import os
results = {}
failed_evaluations = []

def evaluate_model(model_name: str, augmentation_name: str, weight_type: str):
    try:
        if model_name not in model_lookup:
            raise ValueError(f"Model {model_name} not found. Available models: {list(model_lookup.keys())}")
        if augmentation_name not in model_lookup[model_name]:
            raise ValueError(f"Augmentation {augmentation_name} not found for model {model_name}. Available augmentations: {list(model_lookup[model_name].keys())}")
        if weight_type not in model_lookup[model_name][augmentation_name]["weights"]:
            raise ValueError(f"Weight type {weight_type} not found for model {model_name} with augmentation {augmentation_name}. Available weight types: {list(model_lookup[model_name][augmentation_name]['weights'].keys())}")

        weights_path = model_lookup[model_name][augmentation_name]["weights"][weight_type]
        
        # Validate weights path exists
        if not weights_path or not os.path.exists(weights_path):
            raise FileNotFoundError(f"Checkpoint file not found at: {weights_path}")
        
        dataset = model_lookup[model_name][augmentation_name].get("dataset")
        if dataset is None:
            raise ValueError(f"Dataset not configured for {model_name}-{augmentation_name}")

        if model_name == "resnet":
            model = ResNet50(num_classes=dataset.get_num_classes(), weights=ResNet50_Weights.IMAGENET1K_V2)
        elif model_name == "swintransformer":
            model = SwinTransformer(model_select=SwinModelSelect.SWIN_V2_T, num_classes=dataset.get_num_classes(), weights=Swin_V2_T_Weights.IMAGENET1K_V1)
        elif model_name == "myModel":
            model = CustomModel(input_shape=(3, 224, 224), num_classes=dataset.get_num_classes())
        else:
            raise ValueError(f"Unsupported model name: {model_name}")

        model.load_checkpoint(weights_path)
        model.eval()  # Explicitly set model to evaluation mode
        model = model.to("cuda") 
        data_loader = SimpleDataLoader(dataset=dataset, batch_size=64, shuffle=False)
        evaluator = ClassificationEvaluator(num_classes=dataset.get_num_classes(), topk=(1, 5))
        engine = WandbTrainer(
            optimizer=None,  # No training, just evaluation
            evaluator=evaluator,
            epochs=0,
            project="cv-course",
            run_name=f"evaluation-{model_name}-{augmentation_name}-{weight_type}",
            device="cuda",
            use_wandb=False  # Disable wandb logging for evaluation
        )
        run_results = engine.validate(model=model, val_loader=data_loader)
        results[(model_name, augmentation_name, weight_type)] = run_results
        print(f"✓ Evaluation results for {model_name} with {augmentation_name} augmentation and {weight_type} weights: {run_results}")
        
    except Exception as e:
        error_msg = f"{model_name}-{augmentation_name}-{weight_type}: {str(e)}"
        failed_evaluations.append(error_msg)
        print(f"✗ FAILED {error_msg}")
#model_lookup = model_lookup_best_model  # Use the best model lookup for evaluation
for model_name in model_lookup:
    for augmentation_name in model_lookup[model_name]:
        for weight_type in model_lookup[model_name][augmentation_name]["weights"]:
            evaluate_model(model_name, augmentation_name, weight_type)

print(f"\n=== Evaluation Summary ===")
print(f"Successful: {len(results)}")
print(f"Failed: {len(failed_evaluations)}")
if failed_evaluations:
    print("\nFailed evaluations:")
    for failure in failed_evaluations:
        print(f"  - {failure}")

## Visualisation of evaluation results

The following sections produce a suite of visualizations to aid interpretation of evaluation results, enabling quick visual comparison of model performance across augmentations and configurations. Plots include aggregated bar charts comparing accuracy and F1-scores across models and augmentations, per-model faceted views showing effect of different augmentations, and detailed per-class heatmaps revealing class-specific strengths and weaknesses. Multiple perspectives (aggregate vs. per-class, grouped by model vs. by augmentation) help identify which combinations are most robust and which classes remain problematic. All plots are saved to disk and displayed inline, supporting both interactive exploration during analysis and archival for documentation and reporting.

In [ ]:
rename_dict = {
    "base": "Base",
    "sobel": "Sobel Filter",
    "hist_eqal": "Histogram Equalization",
    "normalisation": "Contrast & Brightness Normalization",
    "traffic_boost": "Traffic Sign Color Boost",
    "edge_boost_str_2_sig_7_0": "Edge Sharpening (str=2, sig=7.0)",
    "augmented": "Augmented (Random Crop, Color Jitter, Blur, Rotate)",
    "augmented-balanced": "Augmented + Balanced Sampling",
    "weighted-loss-augmented-balanced": "Augmented + Weighted Loss",
    "hist_equal_augmented_weighted_loss": "Histogram Equalization + Augmented + Weighted Loss",
    "hist_equal_augmented_balanced": "Histogram Equalization + Augmented + Balanced Sampling",
}

model_rename_dict = {
    "resnet": "ResNet-50",
    "swintransformer": "Swin Transformer V2-T",
    "myModel": "Custom CNN + Global Attention",
}
print(results)

In [ ]:
# Grouped bar charts per model: dataset groups, columns for weight types
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

# Toggle: when True plots are shown inline; when False charts are saved to disk
PLOT_INLINE = False  # set to False to save PNGs instead of displaying

# Requested palette
SPACE_INDIGO = "#2b2d42"
LAVENDER_GREY = "#8d99ae"
PLATINUM = "#edf2f4"
BAR_COLORS = [SPACE_INDIGO, LAVENDER_GREY, PLATINUM]

sns.set(style="whitegrid")

# Helper to flatten nested dicts
def flatten_dict(d, parent_key="", sep="_"):
    items = {}
    if isinstance(d, dict):
        for k, v in d.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k
            if isinstance(v, dict):
                items.update(flatten_dict(v, new_key, sep=sep))
            else:
                items[new_key] = v
    return items

label_map = globals().get("rename_dict", {
    "base": "Base",
    "sobel": "Sobel Filter",
    "hist_eqal": "Histogram Equalization",
    "normalisation": "Contrast & Brightness Normalization",
    "traffic_boost": "Traffic Sign Color Boost",
    "edge_boost_str_2_sig_7_0": "Edge Sharpening (str=2, sig=7.0)",
    "augmented": "Augmented (Random Crop, Color Jitter, Blur, Rotate)",
    "augmented-balanced": "Augmented + Balanced Sampling",
    "weighted-loss-augmented-balanced": "Augmented + Weighted Loss",
})
print("Length of results:", len(results))
rows = []
for key, metrics in results.items():
    try:
        model_name, aug, wt = key
    except Exception:
        continue
    row = {"model": model_rename_dict.get(model_name, model_name), "augmentation": aug, "weight_type": wt}

    flat = {}
    if isinstance(metrics, dict):
        flat = flatten_dict(metrics)
    elif isinstance(metrics, (list, tuple)):
        for m in metrics:
            if isinstance(m, dict):
                flat.update(flatten_dict(m))
    elif hasattr(metrics, "__dict__"):
        flat = flatten_dict(vars(metrics))

    # Prefer accuracy, fallback to F1 macro, then F1 micro
    f1_macro = None
    f1_micro = None
    acc = None
    for k, v in flat.items():
        nk = ''.join(ch for ch in str(k).lower() if ch.isalnum())
        try:
            val = float(v)
        except Exception:
            continue
        if nk == "accuracy" or nk == "acc":
            acc = val
        if 'f1macro' in nk or 'macrof1' in nk:
            f1_macro = val
        elif 'f1micro' in nk or 'microf1' in nk:
            f1_micro = val

    metric_val = acc if acc is not None else (f1_macro if f1_macro is not None else f1_micro)
    row['metric'] = metric_val
    row['f1_macro'] = f1_macro
    row['f1_micro'] = f1_micro
    row['accuracy'] = acc
    rows.append(row)

if not rows:
    print("No metric rows found in `results`. Run evaluations first.")
else:
    df = pd.DataFrame(rows)
    out_dir = "/home/mozi30/repos/private/CV_VisionStudio/notebooks/plots"
    os.makedirs(out_dir, exist_ok=True)

    models = df['model'].unique()
    saved = []
    font_size = 10
    for model in models:
        mdf = df[df['model'] == model]
        print(f"\nProcessing model: {model} with {len(mdf)} entries")
        if mdf.empty:
            continue
        # pivot so rows are augmentation (dataset), columns are weight_type
        pivot = mdf.pivot_table(index='augmentation', columns='weight_type', values='metric')
        # Ensure consistent column order
        col_order = [c for c in ['pre-training', 'finetune', 'sgd_finetune'] if c in pivot.columns]
        if not col_order:
            col_order = list(pivot.columns)
        pivot = pivot[col_order]

        # Friendly labels from rename_dict when available
        display_labels = [label_map.get(name, name) for name in pivot.index]

        # Plot grouped bar chart
        fig, ax = plt.subplots(figsize=(10, 6))
        fig.patch.set_facecolor(PLATINUM)
        ax.set_facecolor(PLATINUM)
        pivot.plot(kind='bar', ax=ax, width=0.53, color=BAR_COLORS[:len(pivot.columns)], edgecolor=SPACE_INDIGO)
        ax.set_title(f"{model} — Accuracy Score ", fontsize=12)
        ax.set_xlabel("GTSRB Dataset", fontsize=11)
        ax.set_ylabel("Accuracy Score", fontsize=11)
        ax.title.set_color(SPACE_INDIGO)
        ax.xaxis.label.set_color(SPACE_INDIGO)
        ax.yaxis.label.set_color(SPACE_INDIGO)
        ax.tick_params(axis='x', colors=SPACE_INDIGO, labelsize=font_size)
        ax.tick_params(axis='y', colors=SPACE_INDIGO, labelsize=font_size)
        ax.set_xticklabels(display_labels, rotation=45, ha='right', fontsize=font_size - 2)
        for spine in ax.spines.values():
            spine.set_color(SPACE_INDIGO)
        ax.grid(axis='y', color=LAVENDER_GREY, alpha=0.35)
        try:
            ax.set_ylim(0.98, 0.995)  # Set y-axis limits for better comparison
            #ax.set_ylim(0.95, 0.995)
        except Exception:
            pass
        ax.legend(title='Weight Type', fontsize=9, title_fontsize=10, facecolor=PLATINUM, edgecolor=SPACE_INDIGO, loc='upper left', ncol=len(pivot.columns))
        plt.tight_layout()

        out_path = os.path.join(out_dir, f"accuracy_grouped_{model}.png")
        if PLOT_INLINE:
            # Show plot inline in the notebook
            plt.show()
            plt.close(fig)
            print(f"Displayed grouped bar chart for {model}")
        else:
            plt.savefig(out_path, dpi=200)
            plt.close(fig)
            saved.append(out_path)
            print(f"Saved grouped bar chart for {model} -> {out_path}")

    if not PLOT_INLINE and not saved:
        print("No charts produced — check that `metric` values are present in `results`.")
    elif not PLOT_INLINE:
        print(f"Produced {len(saved)} charts.")

    # also save combined CSV for debugging
    csv_path = os.path.join(out_dir, 'results_plot_data.csv')
    df.to_csv(csv_path, index=False)
    print(f"Saved plot input data to {csv_path}")

In [ ]:
# Save `results` summary as CSV
import pandas as pd
import os

OUT_CSV = "/home/mozi30/repos/private/CV_VisionStudio/notebooks/plots/results_summary.csv"

def flatten_dict(d, parent_key='', sep='_'):
    items = {}
    if isinstance(d, dict):
        for k, v in d.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k
            if isinstance(v, dict):
                items.update(flatten_dict(v, new_key, sep=sep))
            else:
                items[new_key] = v
    return items

rows = []
for key, metrics in results.items():
    try:
        model_name, aug, wt = key
    except Exception:
        # skip malformed keys
        continue
    row = {"model": model_name, "augmentation": aug, "weight_type": wt}

    flat = {}
    if isinstance(metrics, dict):
        flat = flatten_dict(metrics)
    elif isinstance(metrics, (list, tuple)):
        for m in metrics:
            if isinstance(m, dict):
                flat.update(flatten_dict(m))
    elif hasattr(metrics, "__dict__"):
        flat = flatten_dict(vars(metrics))

    for k, v in flat.items():
        # normalize key to safe column name
        col = str(k)
        row[col] = v

    rows.append(row)

if rows:
    df = pd.DataFrame(rows)
    # ensure output dir exists
    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
    df.to_csv(OUT_CSV, index=False)
    print(f"Saved results to {OUT_CSV} ({len(df)} rows)")
else:
    print("No result rows found to save. Check `results` content.")

# mark TODO done
from datetime import datetime
print(f"CSV export completed at {datetime.now().isoformat()}")